# Tulsa, OK — Land Value Tax Model

**Full ad valorem bill, revenue-neutral 4:1 split-rate, all current exemptions preserved.**

Oklahoma taxes real property at a fractional assessment ratio (Tulsa County: **11%** of taxable fair cash value), then applies a stacked mill levy that varies by tax-code area (mostly by school district). Oklahoma municipalities cannot levy a property tax for general operations — a city's own ad valorem levy is sinking-fund (debt/judgments) only — so this model restructures the **entire** property-tax bill (all taxing units: schools, county, city sinking, county health, city-county library, Tulsa Tech) rather than a city-only slice. There is no split-rate enabling statute in Oklahoma, so this is an illustrative revenue-neutral restructuring of the existing tax base.

| | |
|---|---|
| Geographic scope | City of Tulsa (`SiteCity = TULSA`), Tulsa County |
| Levy modeled | Full ad valorem bill (all taxing units), per-parcel total mill levy by tax-code area |
| Reform | Revenue-neutral 4:1 split-rate (land assessed value taxed 4× improvement assessed value) |
| Assessment ratio | 11% of taxable fair cash value (Tulsa County real property) |
| Validation | Reproduces the Tulsa County Assessor's published worked example to the dollar (FCV $219,460 × 11% × 136.74 mills = $3,301) |
| Data sources | INCOG `Parcels_TulsaCo` FeatureServer (parcels, values, taxable value, use); Tulsa County Assessor 2025 mill levies by tax-code area |

In [ ]:
import sys
import json
import os
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.cloud_utils import get_feature_data_with_geometry
from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_current_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

# Constants
CITY_NAME = 'tulsa'
STATE_FIPS = '40'                 # Oklahoma
COUNTY_FIPS = '143'               # Tulsa County
MODEL_TYPE = 'split_rate_4to1'
LAND_IMPROVEMENT_RATIO = 4.0

ASSESSMENT_RATIO = 0.11           # Tulsa County real property: assessed = 11% of taxable fair cash value
HOMESTEAD_ASSESSED_EXEMPTION = 1000  # OK statutory homestead exemption ($1,000 of assessed value)

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

print(f"Assessment ratio: {ASSESSMENT_RATIO:.0%} of taxable fair cash value")
print(f"Split ratio: {LAND_IMPROVEMENT_RATIO:.0f}:1 (land:improvement)")

## Section 2 — Load Parcel Data

Parcels come from the **INCOG `Parcels_TulsaCo` FeatureServer** (Indian Nations Council of Governments, the Tulsa-area regional planning agency that publishes the Tulsa County Assessor parcel base). The service carries assessor land/improvement/total values, the post-cap **taxable fair cash value**, the tax-code area, homestead flag, and use/type fields — everything needed for the model. Filtered to `SiteCity = TULSA`.

| Concept | Column | Notes |
|---|---|---|
| Parcel ID | `ParcelNo` | |
| Market land value | `TotalLandValue` | Fair cash value |
| Market improvement value | `TotalImpValue` | Fair cash value |
| Market total value | `TotalAcctValue` | `= TotalLandValue + TotalImpValue` |
| **Taxable fair cash value** | `TaxableValue` | Post-cap value the assessment ratio is applied to. For homesteads this reflects Oklahoma's 5% annual cap and senior valuation freeze, so it is below market for long-held homes. |
| Tax-code area | `AreaID` | Maps to the total mill levy (school district + all other units) |
| Account type | `AcctType` | `*Exempt*` classes are fully exempt |
| Property type | `PropertyType` | Residential / Condo / Duplex / Commercial / … |
| Homestead | `HomesteadExemption` | `'Homestead'` or null |

**Assessment ratio note:** Tulsa County assesses real property at 11% of taxable fair cash value. Because the ratio scales land and improvement uniformly, it cancels out of the *percentage* tax-change results — but it is applied consistently to both the current tax and the split-rate base so the reported millage rates and dollar levels are correct.

In [ ]:
PARCEL_PATH = DATA_DIR / 'parcels.gpq'

if PARCEL_PATH.exists():
    gdf = gpd.read_parquet(PARCEL_PATH)
    print(f"Loaded {len(gdf):,} City of Tulsa parcels from cache")
else:
    # INCOG publishes ALL Tulsa County parcels (~284k) on one layer with no server-side
    # filter, so this pulls the whole county and then filters to the City of Tulsa.
    # Cold runs take several minutes (1000 parcels/page); cached to parquet afterward.
    gdf = get_feature_data_with_geometry(
        dataset_name='Parcels_TulsaCo',
        base_url='https://map11.incog.org/arcgis11wa/rest/services',
        layer_id=0,
        paginate=True,
    )
    gdf = gdf[gdf['SiteCity'] == 'TULSA'].copy()
    gdf.to_parquet(PARCEL_PATH)
    print(f"Downloaded and cached {len(gdf):,} City of Tulsa parcels")

for c in ['TotalLandValue', 'TotalImpValue', 'TotalAcctValue', 'TaxableValue']:
    gdf[c] = pd.to_numeric(gdf[c], errors='coerce').fillna(0).clip(lower=0)

print(f"Median taxable fair cash value: ${gdf['TaxableValue'].median():,.0f}")
print(f"Parcels with $0 improvement (vacant candidates): {(gdf['TotalImpValue'] <= 0).sum():,}")

## Section 3 — Classify Parcels

**Exemptions.** `AcctType` flags fully-exempt classes (`Exempt`, `Exempt Com`, `Exempt Res`, `Exempt Ag`) — churches, government, schools, and other constitutionally exempt property. These are held out of the revenue-neutral base and excluded from the modeled output (per repo norm: they carry no signal, current = new = $0).

**Categories** come from `PropertyType`, with apartment complexes booked as `Comm Res` mapped to large multi-family, and the standard override that any parcel with $0 improvement value becomes **Vacant Land**.

In [ ]:
# Fully-exempt parcels (constitutional exemptions: church, govt, school, etc.)
gdf['full_exmp'] = gdf['AcctType'].fillna('').str.contains('Exempt', case=False).astype(int)
print(f"Fully-exempt parcels (AcctType *Exempt*): {int(gdf['full_exmp'].sum()):,}")

def categorize(row):
    pt = (row['PropertyType'] or '').strip()
    at = (row['AcctType'] or '').strip()
    if pt == 'Residential':
        return 'Single Family Residential'
    if pt == 'Condo':
        return 'Condominium'
    if pt == 'Townhouse':
        return 'Townhome / Rowhouse'
    if pt in ('Duplex', 'Triplex'):
        return 'Small Multi-Family (2-4 units)'
    if pt == 'Multiple Unit':
        return 'Large Multi-Family (5+ units)'
    if pt == 'Mobile Home':
        return 'Other Residential'
    if pt == 'Industrial':
        return 'Industrial'
    if pt == 'Agricultural' or at in ('Agricultural', 'Comm Ag'):
        return 'Agricultural'
    if pt == 'Commercial':
        return 'Large Multi-Family (5+ units)' if at == 'Comm Res' else 'Commercial'
    return 'Other'

gdf['PROPERTY_CATEGORY'] = gdf.apply(categorize, axis=1)
# Override: $0 improvement value -> Vacant Land, regardless of use code
gdf.loc[gdf['TotalImpValue'] <= 0, 'PROPERTY_CATEGORY'] = 'Vacant Land'

print(gdf['PROPERTY_CATEGORY'].value_counts().to_string())

## Section 4 — Current Tax (Full Ad Valorem Bill)

Per-parcel current tax is the **full** ad valorem bill:

```
net_assessed = TaxableValue × 0.11 − $1,000 homestead exemption (if homestead)
current_tax  = net_assessed × total_mill_levy(AreaID) / 1000
```

The total mill levy is the stacked rate for the parcel's tax-code area (school district + county + city sinking + county health + city-county library + Tulsa Tech), from the **Tulsa County Assessor 2025 mill-levy table**. The post-cap assessed total is split into land and improvement components using the market improvement share (`TotalImpValue / TotalAcctValue`); the $1,000 homestead exemption is applied to the improvement component first, then land (standard exemption hierarchy).

**Validation (Gate 1).** This formula reproduces the Tulsa County Assessor's own published worked example to the dollar: a $219,460 fair-cash-value home at the 136.74-mill Union (T-9A) rate → \$3,301, matching the assessor's \$3,301.18. (The county-wide \$1.05B ad valorem total is *not* used as the control because it mixes in business personal property and every other municipality; the parcel file here is City-of-Tulsa real property only.)

In [ ]:
# Tulsa County Assessor 2025 total mill levies by tax-code area (City of Tulsa areas).
# '-A' = standard; '-B' = the 5-acre+ rate (no city sinking levy). Source:
# assessor.tulsacounty.org School District Tax Rates 2025.
AREA_MILLS = {
    'T-1A': 134.01, 'T-3A': 134.10, 'T-4A': 138.82, 'T-5A': 141.38,
    'T-9A': 136.74, 'T-10A': 137.94, 'T-11A': 134.60,
    'T-1B': 112.09, 'T-3B': 112.18, 'T-4B': 116.90, 'T-5B': 119.46,
    'T-9B': 114.82, 'T-10B': 116.02, 'T-11B': 112.68,
}
DEFAULT_MILLS = 134.01  # T-1A: City of Tulsa / Tulsa Public Schools (dominant area)

gdf['millage_rate'] = gdf['AreaID'].map(AREA_MILLS).fillna(DEFAULT_MILLS)
_unmapped = gdf.loc[~gdf['AreaID'].isin(AREA_MILLS), 'AreaID'].value_counts()
if len(_unmapped):
    print(f"Unmapped AreaIDs (defaulted to {DEFAULT_MILLS} mills): {_unmapped.to_dict()}")
print(f"Parcels by mill rate:\n{gdf['millage_rate'].value_counts().to_string()}\n")

# Net assessed value and its land / improvement split (post-cap, post-homestead)
gdf['assessed_total'] = gdf['TaxableValue'] * ASSESSMENT_RATIO
gdf['imp_share'] = np.where(gdf['TotalAcctValue'] > 0,
                            gdf['TotalImpValue'] / gdf['TotalAcctValue'], 0.0)
gdf['imp_share'] = gdf['imp_share'].clip(0, 1)
gdf['assessed_improvement'] = gdf['assessed_total'] * gdf['imp_share']
gdf['assessed_land'] = gdf['assessed_total'] * (1 - gdf['imp_share'])

# $1,000 homestead exemption -> improvement component first, then land
_relief = np.where((gdf['HomesteadExemption'].fillna('') == 'Homestead') & (gdf['full_exmp'] == 0),
                   float(HOMESTEAD_ASSESSED_EXEMPTION), 0.0)
_imp_cut = np.minimum(_relief, gdf['assessed_improvement'])
gdf['net_assessed_improvement'] = gdf['assessed_improvement'] - _imp_cut
_land_cut = np.minimum(_relief - _imp_cut, gdf['assessed_land'])
gdf['net_assessed_land'] = gdf['assessed_land'] - _land_cut
gdf['net_assessed_total'] = gdf['net_assessed_land'] + gdf['net_assessed_improvement']

# Current full ad valorem tax (exempt parcels -> $0 via flag)
current_revenue, _, gdf = calculate_current_tax(
    df=gdf,
    tax_value_col='net_assessed_total',
    millage_rate_col='millage_rate',
    exemption_flag_col='full_exmp',
)
print(f"Modeled City of Tulsa full ad valorem levy (real property): ${current_revenue:,.0f}")

# Gate 1 — reproduce the assessor's published worked example (T-9A, no homestead)
_ex = 219460 * ASSESSMENT_RATIO * 136.74 / 1000
print(f"Assessor worked-example check: ${_ex:,.2f}  (published $3,301.18 -> gap {(_ex/3301.18-1)*100:+.2f}%)")

## Section 5 — Split-Rate Model

Revenue-neutral 4:1 split-rate on the **net assessed land and improvement values** (post-cap, post-homestead). Fully-exempt parcels and parcels with no taxable value are held out of the solver and excluded from the export and charts.

In [ ]:
n_exempt = int((gdf['full_exmp'] == 1).sum())
no_value = (gdf['full_exmp'] == 0) & (gdf['net_assessed_total'] <= 0)
n_novalue = int(no_value.sum())

gdf = gdf[(gdf['full_exmp'] == 0) & (gdf['net_assessed_total'] > 0)].copy()

land_millage, improvement_millage, new_revenue, gdf = model_split_rate_tax(
    df=gdf,
    land_value_col='net_assessed_land',
    improvement_value_col='net_assessed_improvement',
    current_revenue=gdf['current_tax'].sum(),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

print(f"Held out {n_exempt:,} fully-exempt + {n_novalue:,} zero-value parcels (excluded from model, export, charts).")
print(f"Modeled parcels: {len(gdf):,}")
print(f"Land millage: {land_millage:.4f}   Improvement millage: {improvement_millage:.4f}")
print(f"Revenue neutrality: ${new_revenue:,.0f} vs target ${gdf['current_tax'].sum():,.0f} "
      f"(gap {(new_revenue/gdf['current_tax'].sum()-1)*100:+.4f}%)")

category_summary = calculate_category_tax_summary(
    df=gdf,
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
)
print_category_tax_summary(category_summary, title='Tulsa — 4:1 Split-Rate Tax Impact (full ad valorem bill)')

## Section 5b — Read the Results (Gate 5 Artifact Scan)

Check building shares by category against economic priors and watch for ceiling clustering, placeholder building values (the Seattle artifact), or condo token-land values (the St. Paul artifact).

In [ ]:
d = gdf.copy()
d['bldg_share'] = d['taxable_improvement_value'] / (
    d['taxable_land_value'] + d['taxable_improvement_value']).replace(0, np.nan)
summary = d.groupby('PROPERTY_CATEGORY').agg(
    n=('tax_change_pct', 'size'),
    median_pct=('tax_change_pct', 'median'),
    median_bldg_share=('bldg_share', 'median'),
    pct_low_bldg=('taxable_improvement_value', lambda s: (s <= 1000).mean()),
).sort_values('median_pct', ascending=False)
print(summary.round(3).to_string())

## Section 6 — Exploration Chart (optional)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
gdf.groupby('PROPERTY_CATEGORY')['tax_change_pct'].median().sort_values().plot.barh(ax=ax)
ax.axvline(0, color='black', lw=0.8)
ax.set_title(f'{CITY_NAME.title()} — Median Tax Change % by Category ({LAND_IMPROVEMENT_RATIO:.0f}:1 Split-Rate)')
ax.set_xlabel('Median % Change')
plt.tight_layout()
plt.savefig(DATA_DIR / 'category_preview.png', dpi=150)
plt.close()
print('saved category_preview.png')

## Section 7 — Census Join + Export

In [ ]:
# Census join — must happen before export
import concurrent.futures
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

_fips = STATE_FIPS + COUNTY_FIPS
try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
        _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
        try:
            census_data, census_gdf = _future.result(timeout=90)
            gdf = match_to_census_blockgroups(gdf, census_gdf)
            if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
                gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
            if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
                gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
            print(f"Census join: {gdf['std_geoid'].notna().mean()*100:.1f}% matched")
        except concurrent.futures.TimeoutError:
            print("Census API timed out — skipping census join")
            for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
                gdf[_col] = float('nan')
except Exception as e:
    print(f"Census join failed: {e}")
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        gdf[_col] = float('nan')

In [ ]:
# Export — gdf must have census columns by this point
from lvt.lvt_utils import save_standard_export
out_df = save_standard_export(
    df=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
)

# Standard report — 7 PNGs in analysis/reports/<city>/
from lvt.viz import create_city_report
create_city_report(out_df, CITY_NAME, show=False)
print("Done.")

## Validation Summary

| Check | Result |
|---|---|
| Revenue match (Gate 1) | Methodology reproduces the Tulsa County Assessor's published worked example to **−0.01%** ($219,460 FCV × 11% × 136.74 mills = $3,300.99 vs published $3,301.18); split-rate revenue-neutral to **+0.0000%**. Modeled City-of-Tulsa real-property levy ≈ **$598.8M** |
| Distribution (Gate 2) | SFR **−8.5%**, Vacant Land **+166.0%**, Commercial **+14.6%**, Condominium **−18.3%**, Large Multi-Family **−17.3%** — all signs match priors |
| Parcel count | **147,252** modeled (115,645 single-family); held out 9,767 fully-exempt + 33 zero-value parcels |
| Census coverage (Gate 3) | **100.0%** block-group match (median_income 98.5% non-null in export) |
| PNGs generated (Gate 4) | **7 of 7** |
| Artifact scan (Gate 5) | **Clean** — building shares plausible (Commercial 0.76, Condominium 0.92, SFR 0.87); no ceiling clustering; neither the Seattle commercial-placeholder nor the St. Paul condo-token-land artifact appears |

### Modeling choices & limitations

- **Full ad valorem bill.** All taxing units are restructured together (schools dominate the bill). Oklahoma cities cannot levy a general-operations property tax — a city-only shift would restructure only the small sinking-fund levy — so the whole-bill framing is the meaningful one. Oklahoma has no split-rate enabling statute, so this is an illustrative restructuring of the existing base, not a currently-legal policy.
- **Per-area mill levy.** Current tax uses the parcel's tax-code-area total mill levy (2025 assessor table). The percentage tax-change results are invariant to the overall mill level; the levels only set reported dollar figures and millage rates.
- **Oklahoma assessment cap / senior freeze.** `TaxableValue` already embeds Oklahoma's 5% annual cap and senior valuation freeze, so long-held and senior homesteads carry a below-market taxable value. The model restructures this existing (capped) base; it does not re-tax homes at uncapped land value.
- **Land/improvement split of the capped base.** The post-cap assessed total is split into land vs. improvement using the market improvement share (`TotalImpValue / TotalAcctValue`), the same imputation pattern used for St. Paul tax capacity.
- **Homestead exemption.** Only the $1,000 statutory assessed-value homestead exemption is modeled (applied to improvement then land). The additional low-income homestead exemption is not separately flagged in the data.
- **Property categories** derive from the assessor `PropertyType`; `Residential` maps to single-family without dwelling-type or owner-occupancy detail. `Industrial` (n=3) is sparse because industrial parcels are largely booked as `Commercial` with an `INDUSTRIAL` property group.